# Lab 2 — Guided EDA và data-quality pipeline

**Thời lượng:** 90–105 phút  
**Câu hỏi:** Dữ liệu có đủ tin cậy để tính doanh thu? Category/region nào nổi bật? Có key nào không match customer master?

> Lab này trình bày một quy trình có hướng dẫn. Sau đó bạn phải tự viết lại pipeline trong challenge.

In [ ]:
from pathlib import Path
import json
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
assert (ROOT / 'data' / 'customer_orders_raw.csv').exists(), ROOT
OUTPUT = ROOT / 'outputs'
FIGURES = OUTPUT / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
print('Project root:', ROOT)
print('Pandas:', pd.__version__)

## 1. Nạp dữ liệu và xác định unit

Mỗi dòng phải đại diện cho một order; `Order ID` phải unique sau cleaning.

In [ ]:
raw = pd.read_csv(ROOT / 'data' / 'customer_orders_raw.csv')
customers = pd.read_csv(ROOT / 'data' / 'customers.csv')
print('raw shape:', raw.shape)
display(raw.head())
display(raw.dtypes.rename('dtype').to_frame())
assert raw.shape == (52, 10)

## 2. Data-quality profile trước cleaning

In [ ]:
profile = pd.DataFrame({
    'dtype': raw.dtypes.astype(str),
    'missing_count': raw.isna().sum(),
    'missing_pct': raw.isna().mean().mul(100).round(2),
    'unique_count': raw.nunique(dropna=True),
})
display(profile)
print('duplicate Order ID values:', raw['Order ID'].astype('string').str.strip().duplicated().sum())
print('raw missing cells:', int(raw.isna().sum().sum()))
for column in ['Category', 'Region', 'Payment Method']:
    print(f'\n{column}:', sorted(raw[column].dropna().astype(str).unique()))

**Nhận xét cần ghi:** cột nào bị đọc sai dtype? Nhãn nào chỉ khác hoa/thường/khoảng trắng? Có bao nhiêu missing và duplicate key?

## 3. Các helper thuần túy

In [ ]:
def to_snake_case(name: str) -> str:
    value = re.sub(r'[^0-9a-zA-Z]+', '_', str(name).strip())
    return value.strip('_').lower()

def clean_text(series: pd.Series) -> pd.Series:
    return series.astype('string').str.strip().replace('', pd.NA)

def parse_mixed_dates(series: pd.Series) -> pd.Series:
    text = series.astype('string').str.strip()
    parsed = pd.Series(pd.NaT, index=series.index, dtype='datetime64[ns]')
    iso_dash = text.str.match(r'^\d{4}-\d{2}-\d{2}$', na=False)
    iso_slash = text.str.match(r'^\d{4}/\d{2}/\d{2}$', na=False)
    other = ~(iso_dash | iso_slash)
    parsed.loc[iso_dash] = pd.to_datetime(text.loc[iso_dash], format='%Y-%m-%d', errors='coerce')
    parsed.loc[iso_slash] = pd.to_datetime(text.loc[iso_slash], format='%Y/%m/%d', errors='coerce')
    parsed.loc[other] = pd.to_datetime(
        text.loc[other], format='mixed', dayfirst=True, errors='coerce'
    )
    return parsed

assert to_snake_case(' Order ID ') == 'order_id'

## 4. Normalize schema và text

Tạo deep copy để không mutate `raw`.

In [ ]:
df = raw.copy(deep=True)
df.columns = [to_snake_case(c) for c in df.columns]
text_columns = ['order_id', 'customer_id', 'product', 'category', 'region', 'payment_method']
for column in text_columns:
    df[column] = clean_text(df[column])
df['order_id'] = df['order_id'].str.upper()
df['customer_id'] = df['customer_id'].str.upper()
df['product'] = df['product'].str.replace(r'\s+', ' ', regex=True)
df['category'] = df['category'].str.lower()
print(df.columns.tolist())
pd.testing.assert_frame_equal(raw, pd.read_csv(ROOT / 'data' / 'customer_orders_raw.csv'))

## 5. Parse date/numeric và ghi nhận lỗi

Parse không phải cleaning policy. Trước hết biến giá trị không parse được thành missing, sau đó áp dụng rule.

In [ ]:
df['order_date'] = parse_mixed_dates(df['order_date'])
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')
df['unit_price'] = pd.to_numeric(df['unit_price'], errors='coerce')
df['discount_pct'] = pd.to_numeric(df['discount_pct'], errors='coerce').fillna(0.0)
print('invalid dates:', int(df['order_date'].isna().sum()))
print('non-positive/non-integer quantity:', int((df['quantity'].isna() | (df['quantity'] <= 0) | (df['quantity'] % 1 != 0)).sum()))
print('invalid price:', int((df['unit_price'].isna() | (df['unit_price'] <= 0)).sum()))
print('invalid discount:', int((~df['discount_pct'].between(0, 100)).sum()))

## 6. Canonical mappings

In [ ]:
region_map = {
    'hanoi': 'north', 'ha noi': 'north',
    'da nang': 'central', 'danang': 'central',
    'hcm': 'south', 'hcmc': 'south', 'tp.hcm': 'south',
    'ho chi minh': 'south', 'ho chi minh city': 'south',
}
payment_map = {
    'card': 'card', 'cash': 'cash',
    'e-wallet': 'e_wallet', 'ewallet': 'e_wallet',
    'bank transfer': 'bank_transfer',
}
df['region'] = df['region'].str.lower().map(region_map)
df['payment_method'] = df['payment_method'].str.lower().map(payment_map)
print('unmapped region:', int(df['region'].isna().sum()))
print('unmapped payment:', int(df['payment_method'].isna().sum()))

## 7. Validity mask, duplicate policy và features

In [ ]:
df = df.drop_duplicates(subset='order_id', keep='first')
valid = (
    df[['order_id', 'order_date', 'customer_id', 'product', 'category']].notna().all(axis=1)
    & df['category'].isin({'electronics', 'home', 'office', 'fashion'})
    & df['quantity'].notna() & (df['quantity'] > 0) & np.isclose(df['quantity'] % 1, 0)
    & df['unit_price'].notna() & (df['unit_price'] > 0)
    & df['discount_pct'].between(0, 100)
    & df['region'].notna()
    & df['payment_method'].notna()
)
cleaned = df.loc[valid].copy()
cleaned['quantity'] = cleaned['quantity'].astype('int64')
cleaned['gross_revenue'] = cleaned['quantity'] * cleaned['unit_price']
cleaned['net_revenue'] = cleaned['gross_revenue'] * (1 - cleaned['discount_pct'] / 100)
cleaned = cleaned.sort_values(['order_date', 'order_id']).reset_index(drop=True)
print('cleaned shape:', cleaned.shape)
display(cleaned.head())
assert cleaned.shape == (41, 12)
assert cleaned['order_id'].is_unique
assert cleaned['discount_pct'].between(0, 100).all()

## 8. Quality report và lưu artifact

In [ ]:
duplicate_ids = int(raw['Order ID'].astype('string').str.strip().str.upper().duplicated().sum())
quality_report = {
    'raw_rows': int(len(raw)),
    'clean_rows': int(len(cleaned)),
    'removed_rows': int(len(raw) - len(cleaned)),
    'duplicate_order_ids': duplicate_ids,
    'missing_cells_raw': int(raw.isna().sum().sum()),
}
print(json.dumps(quality_report, indent=2))
cleaned.to_csv(OUTPUT / 'cleaned_orders.csv', index=False)
(OUTPUT / 'data_quality_report.json').write_text(json.dumps(quality_report, indent=2), encoding='utf-8')

## 9. EDA summaries

In [ ]:
category_summary = (
    cleaned.groupby('category', as_index=False)
    .agg(orders=('order_id', 'nunique'), units=('quantity', 'sum'),
         gross_revenue=('gross_revenue', 'sum'), net_revenue=('net_revenue', 'sum'))
    .sort_values('net_revenue', ascending=False)
)
region_summary = (
    cleaned.groupby('region', as_index=False)
    .agg(orders=('order_id', 'nunique'), net_revenue=('net_revenue', 'sum'))
    .sort_values('net_revenue', ascending=False)
)
display(category_summary)
display(region_summary)
print('gross total:', cleaned['gross_revenue'].sum())
print('net total:', cleaned['net_revenue'].sum())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(category_summary['category'], category_summary['net_revenue'])
axes[0].set(title='Net revenue by category', xlabel='Category', ylabel='Net revenue')
axes[1].bar(region_summary['region'], region_summary['orders'])
axes[1].set(title='Orders by region', xlabel='Region', ylabel='Unique orders')
fig.tight_layout()
fig.savefig(FIGURES / 'week01_eda_summary.png', dpi=150)
plt.show()

## 10. Merge với customer master

Dùng `validate` để biến giả định cardinality thành kiểm tra.

In [ ]:
enriched = cleaned.merge(
    customers, on='customer_id', how='left', validate='many_to_one', indicator=True
)
unmatched = enriched.loc[enriched['_merge'] != 'both', ['order_id', 'customer_id', '_merge']]
display(unmatched)
assert len(enriched) == len(cleaned)
assert unmatched[['order_id', 'customer_id']].values.tolist() == [['O026', 'C999']]

## 11. Kết luận của bạn

Viết ít nhất:

1. Ba vấn đề data quality và rule xử lý.
2. Ba insight có số liệu.
3. Hai limitation của dataset.
4. Ba test/invariant cần giữ khi dữ liệu mới đến.

In [ ]:
print('Lab 2 checks: PASS')